# Drug Verification — Formal Verification

Runs `vehicle verify` against the committed `pk.onnx` model using Marabou.
No training happens here — this notebook verifies the canonical model produced by `pk train`.

**Branch:** `feat/jess-suggested-vclScalar`  
**Repo:** [lstrsrmn/drug-verification](https://github.com/lstrsrmn/drug-verification)

### Properties verified

| Property | What it checks |
|---|---|
| `nonNeg` | Output dose is always ≥ 0 |
| `safeNear` | When concentration is near C_safe, the network prescribes a near-zero dose |
| `safeFar` | When concentration is far from C_safe, the network does not push it over the limit |

---

### Prerequisites
- The `pk.onnx`, `pk_mean.idx`, `pk_std.idx`, and `pk.vcl` files must all be present.
  These are committed to the repo and produced by `pk train` locally.
- `vehicle` CLI and Marabou must be available (installed automatically in Colab below).

## 1. Setup

In [ ]:
import os, sys

# Clone the repo if running in Colab
if 'google.colab' in sys.modules:
    if not os.path.exists('drug-verification'):
        !git clone --branch feat/jess-suggested-vclScalar https://github.com/lstrsrmn/drug-verification.git
    os.chdir('drug-verification')

    # Install vehicle-lang for the `vehicle` CLI
    !pip install -q vehicle-lang

print('Working directory:', os.getcwd())

In [ ]:
import glob, shutil

if 'google.colab' in sys.modules:
    _py = f'cp{sys.version_info.major}{sys.version_info.minor}'
    _wheel_versions = {'cp38', 'cp39', 'cp310', 'cp311'}
    if _py in _wheel_versions:
        MARABOU_WHL = (
            f'https://github.com/NeuralNetworkVerification/Marabou/releases/download/v2.0.0/'
            f'maraboupy-2.0.0-{_py}-{_py}-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
        )
        print(f'Installing maraboupy wheel for {_py}')
        !pip install -q {MARABOU_WHL}
    else:
        _repo = os.path.join(os.getcwd(), 'Marabou-src')
        _build = os.path.join(_repo, 'build')
        print(f'No maraboupy wheel for {_py} — building Marabou from source (this takes ~20 min)...')
        !apt-get install -qq cmake
        if not os.path.exists(_repo):
            !git clone -q --depth 1 --branch v2.0.0 https://github.com/NeuralNetworkVerification/Marabou.git {_repo}
        !cmake -S {_repo} -B {_build} -DCMAKE_BUILD_TYPE=Release -DBUILD_PYTHON=OFF -DRUN_UNIT_TEST=OFF -DRUN_REGRESS_TEST=OFF -DRUN_SYSTEM_TEST=OFF
        !cmake --build {_build} --target Marabou -j$(nproc) 2>&1 | tail -5

        _candidates = glob.glob(os.path.join(_build, '**', 'Marabou'), recursive=True)
        _binaries = [p for p in _candidates if os.path.isfile(p) and os.access(p, os.X_OK)]
        if not _binaries:
            raise RuntimeError(f'Marabou binary not found after build. Searched: {_build}')
        shutil.copy2(_binaries[0], '/usr/local/bin/Marabou')
        os.chmod('/usr/local/bin/Marabou', 0o755)
        print('Marabou installed to /usr/local/bin/Marabou')
else:
    print('Not in Colab — assuming vehicle and Marabou are already on PATH')

## 2. Pre-flight checks

All four files must be present before verification can run.

In [ ]:
import subprocess

REQUIRED = ['pk.onnx', 'pk.vcl', 'pk_mean.idx', 'pk_std.idx']

all_present = True
for f in REQUIRED:
    exists = os.path.exists(f)
    print(f'  {"✓" if exists else "✗ MISSING"}  {f}')
    if not exists:
        all_present = False

vehicle_ok = shutil.which('vehicle') is not None
print(f'  {"✓" if vehicle_ok else "✗ MISSING"}  vehicle CLI')

if not all_present:
    print()
    print('ERROR: missing files. Run `pk train` locally and commit pk_mean.idx / pk_std.idx.')
elif not vehicle_ok:
    print()
    print('ERROR: vehicle not found. Install vehicle-lang or add it to PATH.')
else:
    print()
    print('All checks passed — ready to verify.')

## 3. Verification

In [ ]:
PROPERTIES = ['nonNeg', 'safeNear', 'safeFar']
PARAMS = [
    '-p', 'Ka:4.5',
    '-p', 'Ke:3.5',
    '-p', 'Vd:10',
    '-p', 'C_safe:30',
    '-p', 'ttd:2',
    '-p', 'Ka_over:0.3228',
    '-p', 'Ka_under:0.3227',
    '-p', 'Ke_over:0.415',
    '-p', 'Ke_under:0.4149',
    '-p', 'eps:0.001',
]

results = {}
for prop in PROPERTIES:
    cmd = [
        'vehicle', 'verify',
        '-v', 'Marabou',
        '-s', 'pk.vcl',
        '-n', 'pk:pk.onnx',
        '-c', 'cache',
        '-d', 'meanScalingValues:pk_mean.idx',
        '-d', 'standardDeviationValues:pk_std.idx',
        '--property', prop,
    ] + PARAMS

    print(f'Verifying {prop} ...')
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    print(output)
    print()

    passed = 'proved no counterexample exists' in output
    results[prop] = 'PASS ✓' if passed else 'FAIL ✗'

print('=' * 40)
print('Verification summary')
print('=' * 40)
for prop, status in results.items():
    print(f'  {prop:12s}  {status}')